# Task 4: Context-Aware Chatbot Using LangChain and RAG

In [ ]:
!pip install langchain langchain-community chromadb sentence-transformers streamlit pypdf -q

## Load Documents

In [ ]:
from langchain_community.document_loaders import TextLoader
loader=TextLoader('knowledge_base.txt')
docs=loader.load()

## Create Embeddings and Vector Store

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks=splitter.split_documents(docs)
emb=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore=Chroma.from_documents(chunks,emb)

## Retrieval

In [ ]:
retriever=vectorstore.as_retriever(search_kwargs={'k':3})

## Conversational Memory

In [ ]:
from langchain.memory import ConversationBufferMemory
memory=ConversationBufferMemory(return_messages=True)

## Chat Function

In [ ]:
def answer(query):
    docs=retriever.get_relevant_documents(query)
    context=' '.join([d.page_content for d in docs])
    memory.save_context({'input':query},{'output':context[:200]})
    return context[:500]

print(answer('What is this knowledge base about?'))

## Streamlit App

In [ ]:
streamlit_code='''\nimport streamlit as st\nst.title("RAG Chatbot")\nq=st.text_input("Ask")\nif q:\n st.write(answer(q))\n'''
open('app.py','w').write(streamlit_code)

## Final Summary
RAG chatbot with embeddings, vector database, retrieval and conversation memory.